In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

/Users/emc2/Проекты/личное/Практикум/Deep learning Engineer/practicum_dle/Sprint 2/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# загрузка модели и токенизатора
model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 1) Общая конфигурация
cfg = model.config
print("Model config:")
print(f"d_model={cfg.d_model}, num_layers={cfg.num_layers}, feed_forward_size={cfg.d_ff}")
print()

# 2) Блок энкодера
first_encoder_block = model.encoder.block[0]
print("Encoder block:")
print(first_encoder_block)
print()

# 3) Декодер — блоки с self-attention и cross-attention
first_decoder_block = model.decoder.block[0]
print("Decoder block:")
print(first_decoder_block)
print()

# 4) количество параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 5916.87it/s]


Model config:
d_model=768, num_layers=12, feed_forward_size=3072

Encoder block:
T5Block(
  (layer): ModuleList(
    (0): T5LayerSelfAttention(
      (SelfAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
        (relative_attention_bias): Embedding(32, 12)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): T5LayerFF(
      (DenseReluDense): T5DenseActDense(
        (wi): Linear(in_features=768, out_features=3072, bias=False)
        (wo): Linear(in_features=3072, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (act): ReLU()
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)

Decoder block:
T5Block(
  (layer

In [15]:
import torch
from datasets import load_dataset

# Загружаем подмножество для быстрого теста
dataset = load_dataset("billsum", split="ca_test[:50]")

# Пример
print("Example from dataset:")
print(dataset[0])

# тексты и их референсные суммаризации
texts = [item['text'] for item in dataset]
references = [item['summary'] for item in dataset]

tokenized_texts = []

for text in texts:
    input_text = "summarize: " + text.strip().replace("\n", " ")
    tokenized_input_text = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True, padding="max_length")  # токенизируйте текст

    tokenized_texts.append(tokenized_input_text)  # добавьте токенизированный текст в список

print('Example of tokenized text:')
print(tokenized_texts[0])

Example from dataset:
{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThe Legislature finds and declares all of the following:\n(a) (1) Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations help preserve the memories and incidents of the great hostilities fought by our nation, and preserve and strengthen comradeship among members.\n(2) These veterans’ organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together to find camaraderie and fellowship, share stories, and seek support from people who understand their unique experiences. This aids in the healing process for these returning veterans, and ensures their health and happiness.\n(b) As a result of congressional chartering of these veterans’ organizations, 

In [19]:
import evaluate
from tqdm import tqdm

generated_summaries = []

for inputs in tqdm(tokenized_texts):
    with torch.no_grad():
        summary_ids = model.generate(inputs, max_length=50, min_length=10, length_penalty=2.0, num_beams=4, early_stopping=True)


    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)  # декодируйте саммари с помощью токенизатора
    generated_summaries.append(summary)  # добавьте саммари к списку сгенерированных саммари

rouge = evaluate.load("rouge") # создайте объект для расчёта метрики rouge

# Подсчёт метрик
results =  rouge.compute(predictions=generated_summaries, references=references)  # посчитайте метрику rouge

print('Metrics')
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# пример сгенерированного саммари
print('Summary example:')
print(generated_summaries[0])

100%|██████████| 50/50 [04:01<00:00,  4.84s/it]


Metrics
rouge1: 0.1294
rouge2: 0.0469
rougeL: 0.0989
rougeLsum: 0.1114
Summary example:
the state of california enacts a special tax exemption for veterans’ organizations . the exemption applies to posts or organizations of war veterans, among other attributes . the state board of equalization concludes that
